# COGS 188 - Project Proposal

# Project Description

You have the choice of doing either (1) an AI solve a problem style project or (2) run a Special Topics class on a topic of your choice.  If you want to do (2) you should fill out the _other_ proposal for that. This is the proposal description for (1).

You will design and execute a machine learning project. There are a few constraints on the nature of the allowed project. 
- The problem addressed will not be a "toy problem" or "common training students problem" like 8-Queens or a small Traveling Salesman Problem or similar
- If its the kind of problem (e.g., RL) that interacts with a simulator or live task, then the problem will have a reasonably complex action space. For instance, a wupus world kind of thing with a 9x9 grid is definitely too small.  A simulated mountain car with a less complex 2-d road and simplified dynamics seems like a fairly low achievement level.  A more complex 3-d mountain car simulation with large extent and realistic dynamics, sure sounds great!
- If its the kind of problem that uses a dataset, then the dataset will have >1k observations and >5 variables. I'd prefer more like >10k observations and >10 variables. A general rule is that if you have >100x more observations than variables, your solution will likely generalize a lot better. The goal of training an unsupervised machine learning model is to learn the underlying pattern in a dataset in order to generalize well to unseen data, so choosing a large dataset is very important.
- The project must include some elements we talked about in the course
- The project will include a model selection and/or feature selection component where you will be looking for the best setup to maximize the performance of your AI system. Generally RL tasks may require a huge amount of training, so extensive grid search is unlikely to be possible. However expoloring a few reasonable hyper-parameters may still be possible. 
- You will evaluate the performance of your AI system using more than one appropriate metric
- You will be writing a report describing and discussing these accomplishments


Feel free to delete this description section when you hand in your proposal.

# Names

- Niamh

# Abstract 
In this project I will investigate the optimization of automated train routing and scheduling in a stochastic rail network environment. Disruptions such as weather restrictions and congestion create a dynamic environment where static timetables often fail. My goal is to design an agent to navigate a simulated rail grid to minimize transit delay. My environment will be represented as a procedurally generated grid containing 10,000 state configurations defined by train position, velocity, track switching options, and time step horizons. I will compare a deterministic baseline via A* search, a model-based optimal planner via Dynamic Programming (Value Iteration) with known transition probabilities, and a model-free RL agent using Q-learning to optimize routes under stochastic delays. Performance will be measured through cumulative episodic reward, on-time punctuality rates, and average delay duration.

# Background

Automated railway traffic management is a critical challenge in modern transport engineering, as it requires real-time decision-making under uncertain physical variables and environmental disruptions <a name="higgins"></a>[<sup>[1]</sup>](#higginsnote). Traditional pathfinding in static transportation networks relies heavily on heuristic search algorithms such as A* and Greedy Best-First Search, which guarantee optimal paths under fully observable and deterministic conditions (Russell & Norvig, Chapters 3–4). However, real-world rail environments are inherently stochastic and non-deterministic due to unpredictable signal failures, mechanical wear, and weather-induced speed restrictions <a name="caimi"></a>[<sup>[2]</sup>](#caiminote). To address these dynamic constraints, sequential decision-making problems under uncertainty are formally modeled as Markov Decision Processes (MDPs), where exact optimal solutions can be derived using Dynamic Programming techniques such as Value Iteration when environmental transition models are fully known (Russell & Norvig, Chapter 17; Sutton & Barto, Chapter 4). When transition dynamics are unknown or too complex to compute analytically, agents must transition from model-based planning to model-free reinforcement learning. Temporal Difference (TD) learning frameworks, specifically Q-learning, empower agents to learn optimal action-value functions directly from experiential interactions without an explicit environment model (Sutton & Barto, Chapter 6) <a name="watkins"></a>[<sup>[3]</sup>](#watkinsnote). This project builds a unified comparative framework to evaluate how classical heuristic search, exact dynamic programming, and model-free TD learning scale across a high-dimensional simulated rail transit grid.

# Problem Statement

The problem addressed in this project is the real time routing and speed profile optimization of a train navigating a stochastic rail grid from an origin terminal to a destination terminal under strict time constraints. 
- **Quantifiable:** The problem is formulated as a finite horizon Markov Decision Process (MDP) defined by the tuple $(S, A, P, R, \gamma)$, where states S encompass spatial coordinates (x, y), track gradient, current velocity vector v, track switch configurations, and elapsed time step counters t, resulting in a state space exceeding 10,000 distinct configurations. The action space A consists of discrete control adjustments: {Accelerate, Maintain Speed, Decelerate, Switch Track Left, Switch Track Right, Wait/Hold}.
- **Measurable:** System performance is measured objectively via a reward function that assigns small incremental penalties for time delays ($-1$ per time step), high penalties for safety violations or hazardous track collisions ($-100$), and a large positive reinforcement signal ($+200$) for successfully reaching the destination terminal.
- **Replicable:** The simulation environment is procedurally generated using fixed pseudorandom number generator seeds, ensuring that stochastic delay hazards and weather interruptions can be identically reproduced across all evaluated algorithms.

# Data

This project uses a procedurally generated interactive simulation environment rather than a dataset file, continuously generating experience data through agent environment loops:
- **Obtainment:** Programmatically generated custom Python simulation environment modeling rail physics and track networks.
- **Size and Dimensions:** The environment encompasses 15,000 unique state configurations and over 12 primary variables.
- **Observation Structure:** An observation consists of a localized $5\times 5$ track segment matrix, current velocity magnitude, distance remaining to destination, active signal indicators, and recent weather restriction flags.
- **Critical Variables:** Represented as continuous float tensors for velocity and discrete integer vectors for grid coordinates, switch positions, and temporal horizons.
- **Special Handling:** State discretization bounds are enforced to prevent infinite loops, and reward values normalized to stabilize convergence during Q-table updates.

# Proposed Solution

The proposed solution consists of a three tiered comparative algorithmic framework in Python:
1. **Baseline Model ($A^*$ Search):** Assumes a fully deterministic, completely known environment. Implements $A^*$ graph search utilizing a customized admissible heuristic function (Euclidean distance weighted by maximum track speed limits) to establish the theoretical shortest baseline path and cost.
2. **Optimal Planner (Dynamic Programming - Value Iteration):** Formulates the rail network as an explicit MDP with known stochastic transition probabilities (e.g., a 15% probability of a 1-step weather delay on specific segments). Value iteration systematically updates state values across the entire state space until convergence to derive the mathematically optimal policy.
3. **Reinforcement Learning Agent (Q-Learning):** Strips away the transition model entirely, forcing the agent to learn purely through trial-and-error interactions. Implements an $\epsilon$-greedy exploration strategy to manage the exploration-exploitation tradeoff, updating action-value estimates via off-policy Temporal Difference updates: 
$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s,a)\right]$$ 
The performance of Q-learning and Value Iteration will be benchmarked against the deterministic $A^*$ baseline.

# Evaluation Metrics

Performance will be quantified using three metrics:
1. **Punctuality Rate ($P_{\text{on-time}}$):** The percentage of test episodes where the train successfully reaches the destination terminal within the designated scheduling window.
2. **Cumulative Discounted Return ($G_t$):** The total sum of discounted rewards gathered throughout an execution episode, capturing overall path efficiency and penalty avoidance:
$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$
3. **Average Transit Duration:** The mean number of simulation time steps required to complete the route across 1,000 evaluation test runs.

# Ethics & Privacy

Although simulated rail networks do not process private user data or raise direct consumer privacy concerns, automated transportation systems carry safety implications. If deployed, reinforcement learning models and decision algorithms can hallucinate paths, or suffer from adversarial vulnerability when confronted with rare disruptions (such as extreme weather anomalies). To mitigate these risks, our simulation design integrates fail safe boundary constraints and safety override rules that prevent the agent controller from selecting actions leading to collision/derailment states.

# Team Expectations 

My workflow and accountability standards will proceed as follows:
* **Milestones:** Adhere to weekly blocks such as environment setup, baseline search, MDP planning, Q-learning implement, and reports.
* **Version Control Discipline:** Use Git/ GitHub feature branches for every major algorithmic component to ensure it is reporducible.
* **Documentation:** Maintain clear inline documentation and notes in Jupyter notebooks detailing decisions and parameter tuning

Replace this with something meaningful that is appropriate for your needs. It doesn't have to be something that fits this format.  It doesn't have to be set in stone... "no battle plan survives contact with the enemy". But you need a battle plan nonetheless, and you need to keep it updated so you understand what you are trying to accomplish, who's responsible for what, and what the expected due dates are for each item.

| Target Date | Phase | Planned Tasks & Deliverables |
| :--- | :--- | :--- |
| **Week 1** | Proposal & Setup | Finalize proposal structure, initialize GitHub repository, design core grid simulation architecture. |
| **Week 2** | Environment & Baseline | Implement Python simulation environment with stochastic delay mechanics; build and test $A^*$ search baseline. |
| **Week 3** | Dynamic Programming | Formulate the MDP state/action space and transition matrices; implement Value/Policy Iteration. |
| **Week 4** | Reinforcement Learning | Implement Q-Learning agent, tune hyperparameters ($\alpha$, $\gamma$, $\epsilon$-decay), and run training episodes. |
| **Week 5** | Evaluation & Comparison | Run comparative evaluation scripts across all models; generate performance curves and data tables. |
| **Week 6** | Report Finalization | Draft final project report, compile visualizations, and format submission notebooks. |

# Footnotes
<a name="higginsnote"></a>1.[^](#higgins): Higgins, A. (1998). Scheduling railway traffic—a review. *Operational Research Society*, 49(8), 775-785.<br> 
<a name="caiminote"></a>2.[^](#caimi): Caimi, G., et al. (2012). A model predictive control approach for discrete-continuous optimization in railway traffic management. *Journal of Rail Transport Planning & Management*, 2(4), 95-107.<br>
<a name="watkinsnote"></a>3.[^](#watkins): Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning*, 8(3), 279-292.
